# Group Normalization — Group Normalization (2018)_Papers · Foundations & Optimization_**Normalize each example over a GROUP of its channels — not over the batch — so accuracy stays stable even when the batch is tiny.**---This notebook is a practice scaffold for the **Group Normalization — Group Normalization (2018)** lesson. We build GroupNorm from scratch, check it against PyTorch's own layer, confirm it ignores batch size, and then watch it stay stable where BatchNorm degrades. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorchWe implement GroupNorm from scratch, then verify it three ways: (1) the layer and its match against `nn.GroupNorm`, (2) a hand-checked worked example plus the batch-size-independence check, and (3) dropping it into a tiny conv net to confirm the shapes flow through.

### Step 1 — Build GroupNorm from scratch and check it against PyTorchGroupNorm splits each example's `C` channels into `G` groups, then normalizes each group using the mean and variance computed over that group's channels and all spatial positions — **for that one example only**. There is no batch dimension in the statistics. After normalizing, a learned per-channel scale `gamma` and shift `beta` are applied. The oracle is `nn.GroupNorm`: with matching defaults our output must be identical.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

class MyGroupNorm(nn.Module):
    """GroupNorm from scratch — Wu & He (2018), Eq. 1, 2, 7."""
    def __init__(self, num_groups, num_channels, eps=1e-5):
        super().__init__()
        assert num_channels % num_groups == 0, "C must be divisible by G"
        self.G = num_groups
        self.C = num_channels
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(num_channels))    # per-channel scale
        self.beta = nn.Parameter(torch.zeros(num_channels))    # per-channel shift

    def forward(self, x):                          # x: (N, C, H, W)
        N, C, H, W = x.shape
        xg = x.view(N, self.G, C // self.G, H, W)              # split channels into G groups (Eq. 7)
        mu = xg.mean(dim=(2, 3, 4), keepdim=True)              # per-example, per-group mean  (Eq. 2)
        var = xg.var(dim=(2, 3, 4), unbiased=False, keepdim=True)  # biased variance /m       (Eq. 2)
        xg = (xg - mu) / torch.sqrt(var + self.eps)            # normalize                    (Eq. 1)
        xhat = xg.view(N, C, H, W)
        scaled = xhat * self.gamma.view(1, C, 1, 1)            # per-channel scale
        return scaled + self.beta.view(1, C, 1, 1)            # per-channel shift

# THE ORACLE: my GN must equal nn.GroupNorm.
N, C, H, W, G = 4, 8, 5, 5, 2
x = torch.randn(N, C, H, W)
mine = MyGroupNorm(G, C)
ref = nn.GroupNorm(G, C)          # same default eps=1e-5, gamma=1, beta=0
print("allclose:", torch.allclose(mine(x), ref(x), atol=1e-6))   # expect True

### Step 2 — Recompute the worked example and prove batch-independenceTwo quick checks. First, the worked example: one image with 4 channels and 2 groups, where group 0 holds channels `[1,3]` and `[5,7]` — normalizing should give the familiar `[-1.342, -0.447, 0.447, 1.342]` pattern. Second, the headline property: because the statistics use only the example's own channels, running an image alone or inside a batch of 4 must give **identical** output.

In [ ]:
# Recompute the worked example: 1 image, C=4, G=2, group0 = ch0[1,3], ch1[5,7].
gn = MyGroupNorm(2, 4)
xe = torch.tensor([[[[1., 3.]], [[5., 7.]], [[0., 0.]], [[0., 0.]]]])  # (1,4,1,2)
worked = gn(xe)[0, :2].flatten().tolist()
print("worked xhat group0:", worked)             # ~ [-1.342, -0.447, 0.447, 1.342]

# GN is batch-size-independent: same example -> same output in any batch.
one = MyGroupNorm(G, C)
one.eval()
xa = x[:1]                                  # a single example
big = one(x)[0]                             # its output inside a batch of 4
solo = one(xa)[0]                           # its output alone
print("same in batch of 4 vs 1:", torch.allclose(big, solo, atol=1e-6))  # expect True

### Step 3 — Drop it into a tiny conv netFinally, we confirm `MyGroupNorm` behaves like a normal layer inside a model: a conv, our GroupNorm, a ReLU, global pooling, and a linear head. Feeding a batch of 5 RGB images through it should produce a `(5, 2)` output of class logits — proof the layer composes cleanly.

In [ ]:
# Drop it into a tiny conv net.
net = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1),
    MyGroupNorm(2, 8),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(8, 2),
)
print("net output shape:", net(torch.randn(5, 3, 8, 8)).shape)   # torch.Size([5, 2])

## Visualize the data & results_Train the same tiny conv classifier on the same toy data at shrinking batch sizes, with a BatchNorm layer vs a GroupNorm layer — does BN's error rise as the batch gets tiny while GN stays flat?_

### Step 1 — Build the toy data and a swappable-normalizer trainerWe make a toy dataset whose label depends on the mean of channel 0, then define a `run` helper that builds the *same* tiny conv net but with either a `BatchNorm2d` or a `GroupNorm` layer, trains it at a chosen batch size, and returns the final loss. Everything except the normalizer and the batch size is held fixed, so the comparison is clean.

In [ ]:
import torch
import torch.nn as nn

def make_data(n=512):
    g = torch.Generator().manual_seed(7)
    X = torch.randn(n, 3, 8, 8, generator=g)
    y = (X[:, 0].mean(dim=(1, 2)) > 0).long()   # label from channel-0 patch mean
    X[y == 1] += 0.6
    return X, y

def run(norm, bs, steps=60, lr=0.1):
    torch.manual_seed(0)
    nlayer = nn.BatchNorm2d(8) if norm == "bn" else nn.GroupNorm(4, 8)
    net = nn.Sequential(
        nn.Conv2d(3, 8, 3, padding=1),
        nlayer,
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(8, 2),
    )
    opt = torch.optim.SGD(net.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    X, y = make_data()
    net.train()
    g = torch.Generator().manual_seed(123)
    for _ in range(steps):
        idx = torch.randint(0, X.shape[0], (bs,), generator=g)
        opt.zero_grad()
        loss = lossf(net(X[idx]), y[idx])
        loss.backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        return round(lossf(net(X), y).item(), 4)

### Step 2 — Sweep the batch size and watch BN degradeNow we sweep the batch size from 32 down to 2 and print the final loss for both normalizers side by side. BatchNorm's statistics come from the batch, so they grow noisy as the batch shrinks and its loss climbs; GroupNorm's statistics are per-example, so its loss stays roughly flat. This is the paper's central qualitative claim, reproduced on toy data.

In [ ]:
print("bs   BN      GN")
for bs in [32, 16, 8, 4, 2]:
    bn_loss = run("bn", bs)
    gn_loss = run("gn", bs)
    print(f"{bs:<4} {bn_loss:.4f}  {gn_loss:.4f}")
# BN: 0.0581 0.0563 0.0568 0.0720 0.1372  (rises as batch shrinks)
# GN: 0.0831 0.0831 0.0823 0.0851 0.0869  (flat)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Normalize one GroupNorm group with values $[2,4,6,8]$ ($m=4$), then apply $\gamma=2,\beta=1$. (Ignore $\epsilon$.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Mean: $\mu=(2+4+6+8)/4=5$. — _Center of the group's pooled values._
- Variance: $\sigma^2=((2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2)/4=(9+1+1+9)/4=5$; std $\approx 2.236$. — _Biased spread (divide by $m=4$)._
- Normalize: $[(2-5),(4-5),(6-5),(8-5)]/2.236=[-1.342,-0.447,0.447,1.342]$. — _Mean 0, variance ~1 within the group._
- Scale/shift: $2\cdot[-1.342,-0.447,0.447,1.342]+1=[-1.683,0.106,1.894,3.683]$. — _Learned per-channel $\gamma,\beta$._

**Answer:** $\hat{x}=[-1.342,-0.447,0.447,1.342]$, output $=[-1.683,0.106,1.894,3.683]$. Same normalized shape as the $[1,3,5,7]$ example &mdash; GN is invariant to the group's original center and scale.

</details>

**Problem 2.** Ablation: train the tiny conv net with GroupNorm and with BatchNorm at batch sizes 32 and 2. What do you expect for each as the batch shrinks?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run BN at batch size 32, then at batch size 2; record the final loss each time. — _BN's statistics come from the batch, so they get noisy when the batch is tiny._
- Run GN at batch size 32, then at batch size 2. — _GN's statistics are per example, so the batch size should not matter._
- Compare the two curves (the CODEVIZ chart). — _Same net, same data, same steps &mdash; only the normalizer and batch size differ._

**Answer:** BN's loss rises sharply as the batch shrinks (in our small run, ~0.058 at batch 32 to ~0.137 at batch 2), while GN stays roughly flat (~0.083 at every batch size). At batch size 2 GN now beats BN. This reproduces the paper's qualitative claim: GN is stable across batch sizes where BN degrades.

</details>

**Problem 3.** Why does GroupNorm give the SAME output for an example whether the batch holds 256 images or just 1?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Look at the pooling set $\mathcal{S}_i$ (Eq. 7): it requires $k_N=i_N$. — _Every value averaged shares the example's own batch index._
- Note that no value from any other example ever enters $\mu_i$ or $\sigma_i$. — _So the statistics depend only on this one example's activations._

**Answer:** Because $\mathcal{S}_i$ is confined to the example's own channels and pixels, the mean and variance &mdash; and thus the normalized output &mdash; are computed entirely within that one example. Other examples in the batch have no effect, so the result is identical for any batch size, and train and test behave the same.

</details>